# Gemma 4 31B — Colab C / E / F real matched budget experiment

**No training, Kaggle GPU or official submission.** Same scored Champion prompt, tools, model version, 80 turns, and the six previously validated exploratory tasks. Compare C (5min/60/1024), E (8min/36/1024), F (5min/36/1024) on one Colab host; 18 actual scorer evaluations. Requalify all six with the real scorer on Colab before any inference. This is not a new holdout or proof of Kaggle leaderboard performance.

**Before running:** select a GPU runtime (A100 40/80GB or H100 recommended; T4 is not supported by this conservative gate), enable Colab secret `KAGGLE_API_TOKEN` for your own account, and accept the competition/model access terms on Kaggle. Run all cells. No secrets or raw patches are uploaded. The official wheelhouse might be incompatible with Colab's OS/Python/CUDA; the notebook stops instead of substituting a different model or evaluator. Large model and official data downloads require substantial disk space. If you already downloaded them, set the three directory overrides in cell 1. Actual Colab execution and resource allocation require your account.

In [ ]:
from pathlib import Path
import os, json, shutil, subprocess, sys, hashlib, re
ROOT=Path("/content/gemma-cef")
ROOT.mkdir(exist_ok=True)
# Optional: set extracted local directories to avoid large duplicate downloads.
DATA_OVERRIDE=""
WHEELHOUSE_OVERRIDE=""
MODEL_OVERRIDE=""
# Only use a persistent private mounted directory if you deliberately choose to.
g=subprocess.run(["nvidia-smi","--query-gpu=name,memory.free,compute_cap","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True)
rows=[x.split(",") for x in g.stdout.strip().splitlines()]
assert len(rows)==1, "This Colab runner requires one GPU"
name,free,cap=[x.strip() for x in rows[0]]
disk=shutil.disk_usage(ROOT).free/2**30
print({"gpu":name,"free_vram_GiB":round(float(free)/1024,2),"disk_GiB":round(disk,2),"python":sys.version.split()[0]})
assert float(free)/1024>=32 and float(cap)>=8, "BLOCKED: choose A100/H100 with >=32 GiB free"
assert disk>= (30 if DATA_OVERRIDE and WHEELHOUSE_OVERRIDE and MODEL_OVERRIDE else 80), "BLOCKED: insufficient disk for assets; supply already-extracted private directories"
print("COLAB_GPU_PREFLIGHT_PASS")


In [ ]:
# Private authorized credentials: never echo the token.
from google.colab import userdata
token=userdata.get("KAGGLE_API_TOKEN")
assert token, "Enable KAGGLE_API_TOKEN in Colab Secrets"
os.environ["KAGGLE_API_TOKEN"]=token
del token
os.environ["KAGGLEHUB_CACHE"]=str(ROOT/"cache")
subprocess.run([sys.executable,"-m","pip","install","-q","kaggle==2.2.5","kagglehub"],check=True)
import kagglehub


In [ ]:
# Fetch only code files from the frozen project commit; no historical raw-data mirror.
import urllib.request
REF="fa5e9ca4ae3d2d7f5d60a054d336b2aa9c0e0489"
paths=["scripts/build_behavioral_probe_timeout_safe_20261123.py","scripts/build_toolcap60_timeout_recovery.py","scripts/build_v2_toolcap_submission.py","scripts/prepare_public_budgetfit_v2.py","scripts/budget_factorial_20261127.py","scripts/render_budget_factorial_real31b_20261127.py","scripts/render_six_gold_noop_cpu_20261126.py","kaggle/verifier_baseline_runtime.py","kaggle/gold_control_sweep.py"]
for p in paths:
    dst=ROOT/p
    dst.parent.mkdir(parents=True,exist_ok=True)
    with urllib.request.urlopen("https://raw.githubusercontent.com/kolihhan/gemma-4-kaggle/"+REF+"/"+p,timeout=60) as response:
        dst.write_bytes(response.read())
    compile(dst.read_text(),p,"exec")
os.chdir(ROOT)
sys.path.insert(0,str(ROOT))
from scripts.build_behavioral_probe_timeout_safe_20261123 import _source_files,_pack,CHAMPION_SHA256
from scripts.budget_factorial_20261127 import change_number
from scripts.render_budget_factorial_real31b_20261127 import gpu_render,one
from scripts.render_six_gold_noop_cpu_20261126 import SAFE_RUNTIME
# Download prior private results only to recover membership; no outcome-based selection.
for name,ref in [("fork","kolihhan/gemma4-fork-v1-paired-20261127"),("source","lynnsakurai/gemma-4-budget-fit-agent-w-bounded-rescue/2")]:
    d=ROOT/name
    d.mkdir(exist_ok=True)
    subprocess.run(["kaggle","kernels","output",ref,"-p",str(d),"-o"],check=True,stdout=subprocess.DEVNULL)
old=json.loads((ROOT/"fork/paired_fork31b_results.json").read_text())
ids=sorted(old["selected_public_dev_ids"])
assert old["status"]=="completed" and len(ids)==len(set(ids))==17
assert hashlib.sha256("|".join(ids).encode()).hexdigest()=="231db5d504930562c4dc78f696a56009635f02bfcf4a79023531d2a3a7eb8bd2"
ids=sorted(ids,key=lambda t:hashlib.sha256(("budget-factorial-exploratory-six-20261127-v1|"+t).encode()).hexdigest())[:6]
assert hashlib.sha256("|".join(ids).encode()).hexdigest()=="014a264ca83f257e12d8714cfa322f03538771f51366a886d8ddae76764a43a6"
source=ROOT/"source/submission.zip"
control=_source_files(source.read_bytes())
assert hashlib.sha256(_pack(control)).hexdigest()==CHAMPION_SHA256
print("EXACT_CHAMPION_AND_SAME_SIX_PASS")


In [ ]:
# Official assets downloaded privately using your own Kaggle permissions.
# Never post these assets or the raw evaluation logs publicly.
DATA=Path(DATA_OVERRIDE or kagglehub.competition_download("gemma-4-developer-agent")).resolve()
WHEELS=Path(WHEELHOUSE_OVERRIDE or kagglehub.dataset_download("metric/gemma-4-developer-agent-wheelhouse")).resolve()
assert (DATA/"tasks.jsonl").is_file() and (DATA/"snapshots").is_dir() and (DATA/"wheels").is_dir(), "Competition files must be extracted in the official layout; set DATA_OVERRIDE if needed"
assert any(WHEELS.glob("*.whl")), "Official wheelhouse must contain extracted wheels"
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
supported=set(sys_tags())
bad=[]
for wh in WHEELS.glob("*.whl"):
    if "cutlass" in wh.name.lower():continue
    try:
        fname=wh.name.replace("cu128","+cu128") if "cu128" in wh.name and "+" not in wh.name else wh.name
        if not (parse_wheel_filename(fname)[3]&supported):bad.append(fname)
    except Exception:
        bad.append(wh.name)
assert not bad, "BLOCKED: official wheelhouse has incompatible Python/platform tags"
WORK=ROOT/"work"
WORK.mkdir(exist_ok=True)
def localize(src):
    for old,new in [
        ('Path("/kaggle/input/competitions/gemma-4-developer-agent")',f"Path({str(DATA)!r})"),
        ('Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")',f"Path({str(WHEELS)!r})"),
        ('Path("/kaggle/working")',f"Path({str(WORK)!r})"),
        ('Path("/kaggle/working/gemma31_six_gold_noop_safe.json")',f"Path({str(WORK/'gold_noop.json')!r})")]:
        src=src.replace(old,new)
    return src
# Requalify the SAME six on this host, using official verifier only.
# CPU verification runs before loading weights or generating any patches.
gold=(ROOT/"kaggle/gold_control_sweep.py").read_text().split('if __name__ == "__main__":',1)[0]
gold=localize(gold+"\n"+SAFE_RUNTIME.replace("__SELECTED_IDS__",repr(ids)))
gold_path=ROOT/"colab_gold.py"
compile(gold,str(gold_path),"exec")
gold_path.write_text(gold)
with (ROOT/"gold_private.log").open("w") as log:
    proc=subprocess.run([sys.executable,str(gold_path)],stdout=log,stderr=subprocess.STDOUT,timeout=3600)
assert proc.returncode==0,"BLOCKED: official scorer setup failed; inspect private gold_private.log locally"
q=json.loads((WORK/"gold_noop.json").read_text())
print({k:v for k,v in q.items() if k!="rows"})
assert q["tasks"]==6 and q["discriminating"]==6 and q["unknown"]==0,"BLOCKED: Colab gold/no-op qualification failed; do not spend inference compute"
qual={"qualified_count":6,"qualified_ids":ids,"gpu":False,"model_inference":False}
(ROOT/"qualification.json").write_text(json.dumps(qual))
print("COLAB_GOLD_NOOP_6_OF_6_PASS")


In [ ]:
# Download EXACT Kaggle model version 2 (no fallback to another quantization).
MODEL=Path(MODEL_OVERRIDE or kagglehub.model_download("google/gemma-4/Other/gemma-4-31b-it-qat-w4a16-ct/2")).resolve()
assert (MODEL/"config.json").is_file(), "Model directory missing config.json"
assert any(MODEL.glob("*.safetensors")), "Model weights not found"
# Reuse the previous proven Kaggle evaluator renderer. Adapt ONLY host paths,
# the three budget configurations, and task order/denominator.
out=ROOT/"rendered"
gpu_render(source,ROOT/"qualification.json",out)
src=(out/"main.py").read_text()
arms={"C":(5,60,1024),"E":(8,36,1024),"F":(5,36,1024)}
payload={}
for arm,(minutes,calls,thinking) in arms.items():
    f=dict(control)
    f["eval_config.yaml"]=change_number(change_number(f["eval_config.yaml"],"max_time_minutes",5,minutes),"max_tool_calls",60,calls)
    f["configs/sampling.yaml"]=change_number(f["configs/sampling.yaml"],"thinking_budget",4096,thinking)
    assert f["agent.yaml"]==control["agent.yaml"] and f["prompts/system.md"]==control["prompts/system.md"]
    payload[arm]=f
ordered=sorted(ids,key=lambda t:hashlib.sha256(("colab-cef-20261127|"+t).encode()).hexdigest())
names=list(arms)
order=[(t,a) for i,t in enumerate(ordered) for a in (names[i%3:]+names[:i%3])]
src,n=re.subn(r"(?m)^FACTORIAL_PAYLOAD = .*$",lambda m:"FACTORIAL_PAYLOAD = "+repr(json.dumps(payload,sort_keys=True)),src)
assert n==1
src,n=re.subn(r"(?m)^ORDER = .*$",lambda m:"ORDER = "+repr(order),src)
assert n==1
src=one(src,'assert len(ORDER)==30 and len(set(ORDER))==30','assert len(ORDER)==18 and len(set(ORDER))==18')
src=one(src,'assert len(records)==30','assert len(records)==18')
src=one(src,'str(index)+"/30"','str(index)+"/18"')
src=one(src,'"max_tool_calls":int(budget["max_tool_calls"])}','"max_tool_calls":int(budget["max_tool_calls"]),"max_time_minutes":float(budget["max_time_minutes"])}')
src=one(src,'cfg.max_time_minutes==5','cfg.max_time_minutes==float(budget["max_time_minutes"])')
src=one(src,'Path(f"/kaggle/input/models/google/gemma-4/other/{MODEL_NAME}/2")',f"Path({str(MODEL)!r})")
src=localize(src)
# Wheelhouse already installed during CPU qualification. Do not reinstall on GPU.
src=one(src,'    install_official_wheelhouse()\n','    # wheelhouse installed by the CPU qualification process\n')
runfile=ROOT/"colab_cef_run.py"
compile(src,str(runfile),"exec")
runfile.write_text(src)
assert len(order)==18 and len(set(order))==18
print("COLAB_CEF_RENDER_PASS",{"arms":arms,"tasks":6,"evaluations":18,"model":"exact Kaggle v2"})


In [ ]:
# Genuine model inference and official local patch verification.
# The private log and checkpoints stay in this temporary runtime.
import time
log_path=ROOT/"gpu_private.log"
started=time.monotonic()
with log_path.open("w") as log:
    proc=subprocess.Popen([sys.executable,str(runfile)],stdout=log,stderr=subprocess.STDOUT)
    while proc.poll() is None:
        p=WORK/"budget_factorial_real31b.json"
        if p.is_file():
            try:
                x=json.loads(p.read_text())
                print("ACTUAL_EVALUATIONS",len(x.get("results",[])),"/18",flush=True)
            except (OSError,json.JSONDecodeError):pass
        if time.monotonic()-started>4*3600:
            proc.terminate()
            proc.wait(timeout=60)
            raise TimeoutError("Colab 4h local cap; incomplete results not a comparison")
        time.sleep(30)
assert proc.returncode==0,"BLOCKED: private runtime error; inspect gpu_private.log"
x=json.loads((WORK/"budget_factorial_real31b.json").read_text())
assert x["status"]=="completed" and x["selected_public_dev_ids"]==ids and len(x["results"])==18
idx={(r["instance_id"],r["arm"]):r for r in x["results"]}
assert set(idx)=={(t,a) for t in ids for a in arms}
assert all(r["status"]=="completed" and type(r["resolved"]) is bool for r in idx.values()),"Incomplete evaluation; no winner"
summary={"platform":"Colab","official_submission":False,"training":False,"reused_cohort":True,"independent_holdout":False,"tasks":6,"evaluations":18,"arms":{}}
for a in arms:
    rows=[idx[t,a] for t in ids]
    summary["arms"][a]={"resolved":sum(r["resolved"] for r in rows),"calls":sum(r["tool_calls"] for r in rows),"seconds":sum(r["duration_seconds"] for r in rows),"rescues_vs_C":sum(idx[t,a]["resolved"] and not idx[t,"C"]["resolved"] for t in ids),"regressions_vs_C":sum(not idx[t,a]["resolved"] and idx[t,"C"]["resolved"] for t in ids)}
safe=ROOT/"colab_cef_metrics.json"
safe.write_text(json.dumps(summary,indent=2,sort_keys=True))
print(json.dumps(summary,indent=2))
from google.colab import files
files.download(str(safe))
